# Example 26.1: Exploring a statistical shape atlas

In this notebook, we will explore a statistical shape atlas derived from cardiac magnetic resonance (CMR) imaging.

Using Principal Component Analysis (PCA), complex 3D heart shapes can be broken down into individual "modes" of variation. We will compute the variance explained by these shape modes and visualize how adjusting the Z-score of different modes physically alters the 3D geometry of the heart.

_Note: This notebook requires the `UKBRVLV_All.h5` dataset file, which provides the biventricular atlas data_. _Ensure this file is downloaded and placed in the same directory as this notebook before running._


## Importing modules and loading the atlas

First, we import the necessary libraries and load the hierarchical data format (`.h5`) file containing the atlas.


In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
import ipywidgets as widgets
from ipywidgets import interact
import seaborn as sns
import h5py as h5

sns.set_style("ticks")
%matplotlib inline

# Load shape atlas
# Note: Ensure 'UKBRVLV_All.h5' is in the same directory as this notebook
file_path = "./"
try:
    pc = h5.File(file_path + "UKBRVLV_All.h5", "r")
    print("Atlas loaded successfully!")
except FileNotFoundError:
    print(
        "Error: 'UKBRVLV_All.h5' not found. Please download it and place it in the notebook directory."
    )

## Visualizing explained variance

A key concept in PCA is that the first few modes explain the vast majority of the shape variation in the population. Let us plot the explained variance and cumulative variance for the first 15 shape modes.


In [ ]:
# Extract the variance explained by the modes
num_modes = 15
explained_variance = pc["EXPLAINED"][0][0:num_modes]
cumulative_variance = np.cumsum(explained_variance)

# Create the figure and axis
fig, ax = plt.subplots(figsize=(8, 5))

# Bar chart for individually explained variance
pc_indices = np.arange(1, num_modes + 1)
ax.bar(
    pc_indices,
    explained_variance,
    alpha=0.7,
    color="dodgerblue",
    label="Explained Variance",
)

# Line plot for cumulative variance
ax.plot(
    pc_indices,
    cumulative_variance,
    color="black",
    marker="o",
    linestyle="dashed",
    label="Cumulative Variance",
)

# Formatting
ax.set_xticks(pc_indices)
ax.set_xlabel("Principal Component (Mode)")
ax.set_ylabel("Shape Variance Explained (%)")
ax.set_title("Variance Explained by Principal Components")
ax.legend(loc="center right")
ax.grid(axis="y", linestyle="--", alpha=0.5)

plt.show()

## Generating and plotting 3D shapes

To reconstruct a specific heart shape from the PCA data, we take the mean shape (`MU`) and add a linear combination of the principal component coefficients (`COEFF`), scaled by the specific mode's eigenvalue (`LATENT`) and our chosen `score` (the Z-score).

The function below generates the 3D coordinates for the end-diastolic (ED) and end-systolic (ES) states.


In [ ]:
def genModel(atlas, mode, score):
    """
    Generates the 3D coordinates for the heart shape based on a specific PCA mode and Z-score.
    """
    # Calculate the shape vector S
    S = np.transpose(atlas["MU"]) + (
        score * np.sqrt(atlas["LATENT"][0, mode]) * atlas["COEFF"][mode, :]
    )

    # The data contains End-Diastole (ED) and End-Systole (ES) points concatenated.
    # We split them and reshape into (N, 3) coordinate matrices [x, y, z].
    N = S.shape[1] // 2
    ed = np.reshape(S[0, :N], (-1, 3))
    es = np.reshape(S[0, N:], (-1, 3))

    return ed, es


# Generate the mean reference shape (Mode 0, Z-score = 0.0)
ed1, es1 = genModel(pc, 0, 0.0)

# Plot the mean shape
fig = plt.figure(figsize=(8, 8))
ax1 = fig.add_subplot(111, projection="3d")
ax1.scatter(
    ed1[:, 0],
    ed1[:, 1],
    ed1[:, 2],
    color="dodgerblue",
    marker=".",
    alpha=0.5,
    label="End-Diastole (ED)",
)
ax1.scatter(
    es1[:, 0],
    es1[:, 1],
    es1[:, 2],
    color="firebrick",
    marker=".",
    alpha=0.5,
    label="End-Systole (ES)",
)

ax1.set_title("Mean Biventricular Shape")
ax1.legend()
plt.tight_layout()
plt.show()

## Interactive shape explorer

Finally, we can use Jupyter Widgets to interactively explore how modifying the Z-score of different modes physically changes the anatomy of the heart model.


In [ ]:
# We use the mean shape calculated above as our fixed reference
@interact(
    mode=widgets.IntSlider(value=0, min=0, max=10, step=1, description="Mode"),
    zscore=widgets.FloatSlider(
        value=3.0, min=-5.0, max=5.0, step=0.1, description="Z-score"
    ),
)
def update_plot(mode, zscore):
    # Generate the dynamically modified shape
    ed2, es2 = genModel(pc, mode, zscore)

    fig = plt.figure(figsize=(14, 6))

    # Subplot 1: Fixed Reference Shape (Mean)
    ax1 = fig.add_subplot(1, 2, 1, projection="3d")
    ax1.scatter(
        ed1[:, 0], ed1[:, 1], ed1[:, 2], color="dodgerblue", marker=".", alpha=0.3
    )
    ax1.scatter(
        es1[:, 0], es1[:, 1], es1[:, 2], color="firebrick", marker=".", alpha=0.3
    )
    ax1.set_title("Reference Shape (Mean)")

    # Lock axes limits to make comparison easier
    ax1.set_xlim([-100, 100])
    ax1.set_ylim([-100, 100])
    ax1.set_zlim([-100, 100])

    # Subplot 2: Interactive Modified Shape
    ax2 = fig.add_subplot(1, 2, 2, projection="3d")
    ax2.scatter(
        ed2[:, 0],
        ed2[:, 1],
        ed2[:, 2],
        color="dodgerblue",
        marker=".",
        alpha=0.6,
        label="ED",
    )
    ax2.scatter(
        es2[:, 0],
        es2[:, 1],
        es2[:, 2],
        color="firebrick",
        marker=".",
        alpha=0.6,
        label="ES",
    )
    ax2.set_title(f"Modified Shape (Mode {mode}, Z-score={zscore:.1f})")

    ax2.set_xlim([-100, 100])
    ax2.set_ylim([-100, 100])
    ax2.set_zlim([-100, 100])
    ax2.legend()

    plt.tight_layout()
    plt.show()